# Chapter 13 — Proper motions and membership with Gaia

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/halo-substructure-tutorial/blob/main/notebooks/13_gaia_membership.ipynb)

**Goal.** Stars of a satellite share not only a position and a CMD but also a motion across the sky. Gaia measures
proper motions for stars down to G ≈ 21. We fit a mixture model — satellite plus field — to positions and proper motions
simultaneously, measure the satellite's systemic proper motion, and assign each star a probability of membership, in the
spirit of Pace & Li (2019, https://arxiv.org/abs/1806.02345). We apply it to Boötes I and to the extremely diffuse Crater 2
(Torrealba et al. 2016, https://arxiv.org/abs/1601.07178), whose stars are spread over more than a degree.

**You will learn**
1. to select distant halo stars in Gaia with astrometric quality, parallax and CMD cuts;
2. to write a mixture likelihood in position and proper motion, using each star's full error covariance;
3. to compute membership probabilities, and why $P({\rm star})$ is not $P({\rm member})$;
4. to compare a measured proper motion with a reference, including the systematic floor of Gaia's astrometry.

In [ ]:
import hashlib, json, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy import stats

DATA_URL = 'https://github.com/vasilybelokurov/halo-substructure-tutorial/releases/download/data-v0.1/'
_local = Path('../data/MANIFEST.json')            # present when you run inside a clone of the repository
MANIFEST = json.loads(_local.read_text() if _local.exists() else urllib.request.urlopen(DATA_URL + 'MANIFEST.json').read())


def load(name, cache=Path('data')):
    """Download <name>.parquet from the tutorial's data release (again if the local copy is stale), check its SHA-256."""
    path = cache / f'{name}.parquet'
    if not path.exists():
        cache.mkdir(exist_ok=True)
        print(f'downloading {path.name} ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != MANIFEST[name]['sha256']:   # stale or damaged copy
        print(f'downloading {path.name} again (checksum mismatch) ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    assert hashlib.sha256(path.read_bytes()).hexdigest() == MANIFEST[name]['sha256'], f'{path} does not match the manifest'
    return pd.read_parquet(path)


def tangent(ra, dec, ra0, dec0):
    """Gnomonic (tangent-plane) projection about (ra0, dec0). Returns (xi, eta) in degrees: xi towards east, eta north."""
    ra, dec, ra0, dec0 = map(np.radians, (ra, dec, ra0, dec0))
    cosc = np.sin(dec0) * np.sin(dec) + np.cos(dec0) * np.cos(dec) * np.cos(ra - ra0)
    xi = np.cos(dec) * np.sin(ra - ra0) / cosc
    eta = (np.cos(dec0) * np.sin(dec) - np.sin(dec0) * np.cos(dec) * np.cos(ra - ra0)) / cosc
    return np.degrees(xi), np.degrees(eta)


lvdb = load('lvdb_pm').set_index('key')                         # LVDB v1.1.1 rows for Boötes I and Crater 2
iso = load('isochrones_gaia')

## 13.1 Selecting candidate members in Gaia

For each satellite we keep Gaia DR3 stars with G < 21 that pass four cuts:

1. **astrometric quality**: RUWE < 1.4 (the astrometric solution fits a single star well);
2. **not obviously a galaxy**: `astrometric_excess_noise` below $10^{0.15(G-15)+0.25}$ mas (Koposov et al. 2017; Chapter 10);
3. **not nearby**: the parallax is consistent with the satellite's distance, $\varpi - 3\sigma_\varpi < 1/D$;
4. **on the CMD sequence**: within 0.1 mag (plus three times the expected colour error) in $G_{\rm BP} - G_{\rm RP}$ of an
   old, metal-poor PARSEC isochrone in Gaia bands at the satellite's distance, or on its horizontal branch.

Gaia magnitudes are not corrected for extinction here; at these high latitudes $E(B-V) \lesssim 0.05$, which shifts
$G_{\rm BP} - G_{\rm RP}$ by ≲ 0.07 mag — within the colour window.

In [ ]:
def select(name, key, age=1.26e10, feh=-2.0):
    ref = lvdb.loc[key]
    d = load(name)
    G, c = d.phot_g_mean_mag.to_numpy(), d.bp_rp.to_numpy()
    I = iso[np.isclose(iso.age, iso.age.unique()[np.argmin(np.abs(iso.age.unique() - age))]) &
            np.isclose(iso.feh, iso.feh.unique()[np.argmin(np.abs(iso.feh.unique() - feh))])]
    rgb = I[I.label.isin([1, 2, 3])].sort_values('gmag')                     # main sequence, subgiants, giant branch
    hb = I[I.label == 4]
    mu = ref.distance_modulus
    c_iso = np.interp(G, rgb.gmag.to_numpy()[::-1] * 0 + np.sort(rgb.gmag.to_numpy() + mu),
                      (rgb.g_bpmag - rgb.g_rpmag).to_numpy()[np.argsort(rgb.gmag.to_numpy())], left=np.nan, right=np.nan)
    c_err = 0.02 + 0.1 * np.exp((G - 20.0) / 1.0)                             # rough BP-RP error of faint stars
    on_rgb = np.abs(c - c_iso) < 0.1 + 3 * c_err
    hb_g = np.median(hb.gmag) + mu
    on_hb = (np.abs(G - hb_g) < 0.4) & (c > -0.4) & (c < 0.6)
    aen_ok = d.astrometric_excess_noise < 10**(0.15 * (G - 15) + 0.25)
    plx_ok = d.parallax - 3 * d.parallax_error < 1 / (10**(mu / 5 + 1) / 1e3)
    keep = (d.ruwe < 1.4) & aen_ok & plx_ok & (on_rgb | on_hb)
    x, y = tangent(d.ra.to_numpy(), d.dec.to_numpy(), ref.ra, ref.dec)
    print(f'{key}: {len(d):,} Gaia stars, {keep.sum():,} pass the cuts')
    return d[keep].reset_index(drop=True), x[keep] * 60, y[keep] * 60, (G, c, keep, c_iso)


boo, bx, by, boo_cmd = select('bootes1_gaia', 'bootes_1')
cra, cx, cy, cra_cmd = select('crater2_gaia', 'crater_2')

## 13.2 The mixture model

Each star has a position $\mathbf x_i$ and a measured proper motion $\boldsymbol\mu_i = (\mu_{\alpha*}, \mu_\delta)_i$ with
covariance $\mathbf C_i$ (from the errors and their correlation `pmra_pmdec_corr`). The likelihood of one star is

$$ \ell_i = f\, p_{\rm sat}(\mathbf x_i)\, \mathcal N(\boldsymbol\mu_i \mid \boldsymbol\mu_{\rm sys}, \mathbf C_i)
          + (1-f)\, p_{\rm bg}(\mathbf x_i)\, p_{\rm bg}(\boldsymbol\mu_i). $$

- $p_{\rm sat}(\mathbf x)$: the elliptical exponential of Chapter 6 with the catalogued centre, $r_h$, ε and PA, held fixed
  (Crater 2: round, $r_h = 31′$).
- All satellite stars share one systemic motion $\boldsymbol\mu_{\rm sys}$; the internal velocity dispersion (a few km/s,
  i.e. ~0.01–0.02 mas/yr at these distances) is negligible next to the errors.
- $p_{\rm bg}(\mathbf x)$ is uniform; $p_{\rm bg}(\boldsymbol\mu)$ is a two-component Gaussian mixture fitted beforehand to the
  stars beyond 3 $r_h$ (their measured proper motions, so errors are included empirically).

The free parameters are $f$ and $\boldsymbol\mu_{\rm sys}$. The membership probability of a star is the satellite term
divided by $\ell_i$.

In [ ]:
def gmm2(v, n_iter=300, seed=0):
    """Two-component 2-d Gaussian mixture (EM). Returns weights, means, covariances."""
    rng = np.random.default_rng(seed)
    w = np.array([0.5, 0.5]); m = v[rng.choice(len(v), 2)]; C = np.array([np.cov(v.T)] * 2)
    for _ in range(n_iter):
        r = np.array([w[k] * stats.multivariate_normal(m[k], C[k]).pdf(v) for k in range(2)])
        r /= r.sum(0)
        w = r.mean(1)
        m = (r @ v) / r.sum(1)[:, None]
        C = np.array([((r[k][:, None] * (v - m[k])).T @ (v - m[k])) / r[k].sum() + 1e-6 * np.eye(2) for k in range(2)])
    return w, m, C


def p_sat_xy(x, y, rh, eps, pa):
    a = rh / 1.678
    t = np.radians(pa)
    u = x * np.sin(t) + y * np.cos(t)
    v = x * np.cos(t) - y * np.sin(t)
    return np.exp(-np.sqrt(u**2 + (v / (1 - eps))**2) / a) / (2 * np.pi * a**2 * (1 - eps))


def fit_members(d, x, y, rh, eps, pa, rmax):
    v = d[['pmra', 'pmdec']].to_numpy()
    sa, sd, rho = d.pmra_error.to_numpy(), d.pmdec_error.to_numpy(), d.pmra_pmdec_corr.to_numpy()
    C = np.stack([np.stack([sa**2, rho * sa * sd], -1), np.stack([rho * sa * sd, sd**2], -1)], -2)    # (N, 2, 2)
    Cinv, logdet = np.linalg.inv(C), np.log(np.linalg.det(C))
    R = np.hypot(x, y)
    wb, mb_, Cb = gmm2(v[R > 3 * rh])
    p_bg_pm = sum(wb[k] * stats.multivariate_normal(mb_[k], Cb[k]).pdf(v) for k in range(2))
    p_bg_xy = 1 / (np.pi * rmax**2)
    ps = p_sat_xy(x, y, rh, eps, pa)

    def terms(q):
        f = 1 / (1 + np.exp(-q[0]))
        dv = v - q[1:3]
        sat = f * ps * np.exp(-0.5 * np.einsum('ni,nij,nj->n', dv, Cinv, dv) - 0.5 * logdet) / (2 * np.pi)
        return sat, sat + (1 - f) * p_bg_xy * p_bg_pm

    nll = lambda q: -np.sum(np.log(terms(q)[1]))
    start = np.median(v[R < rh], axis=0)
    best = minimize(nll, [-3, *start], method='Nelder-Mead', options=dict(xatol=1e-6, fatol=1e-7, maxiter=4000))
    h = np.array([1e-3, 1e-4, 1e-4])
    Hm = np.array([[(nll(best.x + ea + eb) - nll(best.x + ea - eb) - nll(best.x - ea + eb) + nll(best.x - ea - eb)) / (4 * h[i] * h[j])
                    for j, eb in enumerate(np.eye(3) * h)] for i, ea in enumerate(np.eye(3) * h)])
    cov = np.linalg.inv(Hm)[1:, 1:]
    sat, tot = terms(best.x)
    return best.x[1:3], cov, sat / tot, v


results = {}
for key, (d, x, y) in {'bootes_1': (boo, bx, by), 'crater_2': (cra, cx, cy)}.items():
    ref = lvdb.loc[key]
    eps = 0.0 if not np.isfinite(ref.ellipticity) else ref.ellipticity
    pa = 0.0 if not np.isfinite(ref.position_angle) else ref.position_angle
    rmax = 90 if key == 'bootes_1' else 150
    mu_sys, cov, pmem, v = fit_members(d, x, y, ref.rhalf, eps, pa, rmax)
    results[key] = dict(mu=mu_sys, cov=cov, pmem=pmem, v=v, x=x, y=y)
    err = np.sqrt(np.diag(cov))
    print(f'{key}: mu_alpha* = {mu_sys[0]:.3f} ± {err[0]:.3f}, mu_delta = {mu_sys[1]:.3f} ± {err[1]:.3f} mas/yr; '
          f'sum of membership probabilities = {pmem.sum():.0f}; stars with P > 0.5: {(pmem > 0.5).sum()}')

## 13.3 Comparison with the literature, including systematics

The Local Volume Database quotes systematic proper motions from Pace et al. (2022,
https://ui.adsabs.harvard.edu/abs/2022ApJ...940..136P), measured with Gaia EDR3 — essentially the same astrometry, so
agreement tests our *method*, not Gaia. Gaia's proper motions also have spatially correlated systematic errors: averaged
over a small field, they cannot be trusted to better than ~0.025 mas/yr (Vasiliev & Baumgardt 2021,
https://ui.adsabs.harvard.edu/abs/2021MNRAS.505.5978V). We add this floor to both axes and test whether the difference lies
inside the 99% ellipse of the combined covariance ($\chi^2 < 9.21$ for 2 degrees of freedom).

In [ ]:
SYS = 0.025
fig, ax = plt.subplots(2, 3, figsize=(16, 9.5))
for row, key in enumerate(results):
    r, ref = results[key], lvdb.loc[key]
    lit = np.array([ref.pmra, ref.pmdec])
    C_lit = np.diag([ref.pmra_em**2, ref.pmdec_em**2])
    delta = r['mu'] - lit
    chi2 = delta @ np.linalg.inv(r['cov'] + C_lit + SYS**2 * np.eye(2)) @ delta
    r['chi2'] = chi2
    print(f'{key}: this fit {np.round(r["mu"], 3)}, LVDB {lit}; chi2 = {chi2:.2f} (99% limit 9.21)')
    m = r['pmem']
    o = np.argsort(m)
    ax[row, 0].scatter(r['v'][o, 0], r['v'][o, 1], c=m[o], s=4, cmap='viridis', vmin=0, vmax=1)
    ax[row, 0].plot(*lit, 'r+', ms=15, mew=2, label='LVDB (Pace et al. 2022)')
    ax[row, 0].set(xlim=(lit[0] - 3, lit[0] + 3), ylim=(lit[1] - 3, lit[1] + 3), xlabel=r'$\mu_{\alpha*}$ [mas/yr]',
                   ylabel=r'$\mu_\delta$ [mas/yr]', title=f'{key}: proper motions (colour = P(member))')
    ax[row, 0].legend(fontsize=8)
    ax[row, 1].scatter(r['x'][o], r['y'][o], c=m[o], s=4, cmap='viridis', vmin=0, vmax=1)
    lim = 90 if key == 'bootes_1' else 150
    ax[row, 1].set(xlim=(lim, -lim), ylim=(-lim, lim), aspect='equal', xlabel="ξ [′]", ylabel="η [′]", title='positions')
    G, c, keep, c_iso = boo_cmd if key == 'bootes_1' else cra_cmd
    ax[row, 2].scatter(c[keep][o], G[keep][o], c=m[o], s=5, cmap='viridis', vmin=0, vmax=1)
    ax[row, 2].set(xlim=(-0.5, 1.8), ylim=(21, 15.5), xlabel='$G_{\\rm BP} - G_{\\rm RP}$', ylabel='G', title='CMD of selected stars')
plt.tight_layout()
assert all(r['chi2'] < 9.21 for r in results.values())
assert (results['crater_2']['pmem'] > 0.5).sum() > 30

**Figure 13.1.** Selected Gaia stars around Boötes I (top) and Crater 2 (bottom), coloured by membership probability:
proper motions (left; red cross: the published systemic motion), positions (middle) and CMD (right). Members cluster
tightly in proper motion, concentrate on the sky, and trace a giant branch and horizontal branch.

Crater 2 is the instructive case. Its surface brightness is so low that in star counts alone it barely rises above the
field, but its stars share a common motion: the proper motion separates them from the foreground even where the spatial
density gives little leverage. Conversely, Gaia reaches only the brightest giants of distant satellites; for fainter or
more distant systems, deep photometry (Chapters 11–12) must do the work.

A star with high $P({\rm member})$ is not "certainly a member", and a star with $P({\rm star}) \approx 1$ from its image
(Chapter 1) may well be a foreground star with $P({\rm member}) \approx 0$. Membership probabilities depend on the model
(fixed structure, a two-component field model) and should be treated as weights, not labels.

## Summary

- Gaia astrometry adds an independent dimension: a satellite is a clump in proper motion as well as on the sky.
- A mixture model with per-star error covariances measures the systemic proper motion and membership probabilities.
- Agreement with published values must include Gaia's systematic floor (~0.025 mas/yr), and is a test of the method when
  the reference uses the same data.